# E1 Raw Baseline — Dual T4

**Setup**
1. Accelerator: GPU T4 x2
2. Attach SeaClear dataset
3. Internet ON (clone + pip)
4. Set `REPO_URL` / `STAGE` below

Stages: `smoke` (2 epochs) → `prep` → `e1` (full baseline).

In [ ]:
# ========= USER CONFIG =========
REPO_URL = "https://github.com/heller007/underwater.git"
REPO_DIR = "/kaggle/working/underwater"
BRANCH = "main"
STAGE = "smoke"  # smoke | prep | e1
HELD_OUT_SITE = "Lokrum"
# Set explicitly if auto-discovery fails:
SEACLEAR_ROOT = None  # e.g. "/kaggle/input/seaclear-marine-debris-detection-and-segmentation"
DEVICE = "0,1"  # dual T4
# ===============================

In [ ]:
import os, subprocess, sys
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU{i}:", torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 1, "Enable GPU T4 x2 accelerator"
if torch.cuda.device_count() < 2:
    print("WARNING: only 1 GPU visible; dual-T4 expected. Continuing with device=0")

In [ ]:
# Install slim deps only (do NOT reinstall torch)
%pip install -q ultralytics imagehash scikit-image pycocotools opencv-python-headless

In [ ]:
import shutil
from pathlib import Path

if Path(REPO_DIR).exists():
    # Update existing clone
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--all"])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", BRANCH])
else:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repo at", REPO_DIR, "rev", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())

In [ ]:
# Discover / list input datasets
print(" /kaggle/input contents:")
for p in sorted(Path("/kaggle/input").glob("*")):
    print(" -", p)

from src.common import load_env, hardware_info
env = load_env("kaggle")
print("Resolved seaclear_root:", env.seaclear_root)
print("Device:", env.device)
print(hardware_info())

if SEACLEAR_ROOT:
    print("Using override SEACLEAR_ROOT=", SEACLEAR_ROOT)
elif env.seaclear_root is None:
    raise SystemExit("Could not find SeaClear under /kaggle/input — set SEACLEAR_ROOT")

In [ ]:
cmd = [
    sys.executable, "scripts/run_stage.py",
    "--stage", STAGE,
    "--env", "kaggle",
    "--held-out-site", HELD_OUT_SITE,
    "--device", DEVICE if torch.cuda.device_count() >= 2 else "0",
]
if SEACLEAR_ROOT:
    cmd += ["--seaclear-root", SEACLEAR_ROOT]
if STAGE == "smoke":
    cmd += ["--max-images", "100"]

print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=REPO_DIR)

In [ ]:
# List artifacts to download from the notebook Output panel
from pathlib import Path
runs = Path("/kaggle/working/runs")
reports = Path("/kaggle/working/reports")
print("Reports:", list(reports.glob("*"))[:20])
print("Runs:")
for r in sorted(runs.glob("*"))[-5:]:
    print(" ", r)
    for w in r.rglob("*.pt"):
        print("   weight:", w, "size_mb", round(w.stat().st_size/1e6, 2))

## Resume tips

- Re-run the clone cell to pull latest code after a GitHub push.
- Training checkpoints live under `/kaggle/working/runs/.../train/weights/`.
- For long E1 runs, download `best.pt` + `metrics_all.json` before the session expires (~12h).
- Optional: upload a “results” Dataset and copy prior `runs/` into `/kaggle/working` at start to resume.